1. Word2Vec (the big picture)
- Unlike BoW/TF-IDF, which represent words as sparse counts, Word2Vec learns a dense, low-dimensional vector for each word (e.g. 50–300 numbers) by training a tiny neural network to predict words from their context. 
- Words used in similar contexts end up with similar vectors — so the vectors capture meaning, not just frequency.

In [1]:
import pandas as pd

data = pd.read_csv("../data/fd_dataset_messy.csv")
print(data.head(5))

                         email                 time        subject  \
0  mukesh.bhatt@rediffmail.com  2025-02-17 20:29:47  Payment issue   
1      sanjay.jain@hotmail.com  2024-06-07 16:22:21           Help   
2         ashok.nair@gmail.com  2024-04-05 08:16:39     Loan query   
3    manoj.gupta51@hotmail.com  2025-01-04 23:24:05          Query   
4     vinod.chopra87@gmail.com  2025-03-03 10:12:43    Loan status   

                                             content              label  
0  Branch gaya tha, unhone email karne bola. Two ...             Non-FD  
1  Dear customer care, My insurance policy is rea...             Non-FD  
2  Namaskar, Rs 6 lakh kat gaya bina bataye insur...             Non-FD  
3  Dear customer care, 1. Suna hai Bajaj Finance ...  Multiple Category  
4  Dear customer care, Where is my money? The rev...             Non-FD  


In [3]:
import re
from gensim.models import Word2Vec

def tokenize(s):
    return re.findall(r'\b[a-zA-Z]+\b', s.lower())

sentences = [tokenize(t) for t in data['content'].astype(str)]

print("First Email Sample:", data['content'][0])
print("\n")
print("Word2Vec ENCODING")
print(sentences[0])

First Email Sample: Branch gaya tha, unhone email karne bola. Two wheeler loan ki EMI is mahine zyada kati hai. Pehle se Rs 800 jyada. Kyu? Statement bhejo. Bahut pareshan ho gaya hoon. Mukesh Bhatt


Word2Vec ENCODING
['branch', 'gaya', 'tha', 'unhone', 'email', 'karne', 'bola', 'two', 'wheeler', 'loan', 'ki', 'emi', 'is', 'mahine', 'zyada', 'kati', 'hai', 'pehle', 'se', 'rs', 'jyada', 'kyu', 'statement', 'bhejo', 'bahut', 'pareshan', 'ho', 'gaya', 'hoon', 'mukesh', 'bhatt']


Note: Word2Vec needs a list of tokenized sentences (list of lists of words), not raw strings — that's the key input difference from CountVectorizer/TfidfVectorizer.

2. CBOW (Continuous Bag of Words)
- CBOW predicts the middle word from its surrounding context words. 
- E.g. given ["wheeler", "___", "ki", "emi"], it tries to predict "loan". It's faster to train and tends to work fine for frequent words.

In [5]:
cbow_model = Word2Vec(sentences, vector_size=50, window=5, min_count=2, sg=0, epochs=50)
# sg=0 -> CBOW
print(cbow_model.wv.most_similar('loan', topn=5))

[('reasons', 0.4583195447921753), ('outstanding', 0.4182426333427429), ('now', 0.3763711750507355), ('extend', 0.37242430448532104), ('interst', 0.36772462725639343)]


##### Hyperparameter Explaination

Parameter 1 — vector_size=50
- Each word gets represented as a list of 50 numbers (a 50-dimensional vector). 
- This is just the "size" of that fingerprint — bigger vectors can capture more nuance, but need more training data to fill them meaningfully. 
- Your vocabulary only has 706 words after filtering, from 2500 short emails — that's a small corpus, so 50 is already a modest, reasonable choice. 
- Going up to something like 300 (common for huge corpora) would just give the model far more numbers to tune than your data can actually justify.

Parameter 2 — window=5
- This sets how many words on each side count as "context" for a target word. I checked it directly on row 0, predicting "emi":
    - before: ['bola', 'two', 'wheeler', 'loan', 'ki']
    - after:  ['is', 'mahine', 'zyada', 'kati', 'hai']
- With window=5, CBOW uses all 10 of those words to try to predict "emi". If you set window=2, it would only use ['loan', 'ki'] and ['is', 'mahine'] — a much tighter, more local context.

Parameter 3 — min_count=2
- Only words appearing at least twice across the whole corpus get a vector at all, everything rarer gets ignored completely (no vector, can't appear in most_similar, etc). 
- I counted it: your corpus has 902 unique words total, and 196 of them appear exactly once — mostly typos like 'payent', 'sttement', 'screensot', 'uresh'. 
- Those get dropped, leaving 706 words that actually get trained — which matches the vocabulary size you saw earlier (len(cbow_model.wv.index_to_key) = 706).

Parameter 4 — sg=0
- This is the architecture switch: 
    - sg=0 → CBOW (predict the middle word from its context — what we've been doing). 
    - sg=1 → Skip-gram (predict the context from the middle word — the one we compared it against earlier).

Parameter 5 — epochs=50
- This is how many full passes the model makes over your entire dataset before stopping. 
- Each pass slides the context window over every word in every one of your 2500 emails and nudges the vectors slightly closer to "correct." 50 full passes means it sees every sentence 50 times. 
- On a small, repetitive dataset like yours, more epochs means the model leans even harder into whatever patterns repeat most — which is part of why "loan" ended up so tightly bound to words from its repeated template sentences ("outstanding", "extend") rather than broader synonyms.

Quick rule of thumb if you want to tune this later:
- lower min_count keeps more rare/typo words (noisier vectors)
- higher window captures broader topic context but blurs local word order
- more epochs sharpens patterns but risks overfitting on a small/templated corpus like this one.

##### Output Explaination

cbow_model.wv.most_similar('loan', topn=5)

-  .wv: This stands for "word vectors." It's the part of the trained model that actually stores every word's 50-number vector and knows how to compare them. (cbow_model itself holds training settings/state; cbow_model.wv holds just the learned vectors and lookup/similarity tools.)
-  .most_similar('loan', ...): This takes "loan"'s 50-dimensional vector, compares it against the vector of every other word in the vocabulary using cosine similarity, and sorts the results from most similar to least. I checked — your vocabulary has 706 words, so this is comparing "loan" against the other 705 candidates.
- topn=5: This just caps how many results come back. topn=5 → top 5 matches. If you drop the argument entirely, gensim defaults to topn=10:

Note 1 — What CBOW is actually trying to learn
- For every word in every email, CBOW looks at the words around it (the "context window") and tries to predict the missing middle word. 
- Example from your data: given context ["wheeler", "ki", "emi"], it tries to guess the missing word is "loan". 
- It does this for every word, in every one of your 2500 emails, repeatedly (you set epochs=50, so it goes over the whole dataset 50 times).


Note 2 — Where the "vector" actually comes from
- Internally, this prediction task is done by a small neural network with one hidden layer of size 50 (vector_size=50). 
- To get good at predicting target words from context, the network is forced to assign each word a 50-number "fingerprint." 
- There's no human-designed meaning behind any of the 50 numbers — they're just whatever pattern the network settled on while trying to get good at the fill-in-the-blank task. 
- The word vector is a side-effect of training, not the goal itself.

Note 3 — What most_similar('loan') actually computes

Once training is done, every word has its own 50-number vector. most_similar just measures the angle (cosine similarity) between "loan"'s vector and every other word's vector, and returns the closest ones. So your output:
[('reasons', 0.458), ('outstanding', 0.418), ('now', 0.376), ('extend', 0.372), ('interst', 0.368)]
means: after 50 rounds of fill-in-the-blank training, the vector the network landed on for "loan" happens to point in roughly the same direction as the vectors for "reasons", "outstanding", "now", "extend", and "interst" (misspelling of "interest").
Note 4 — Why specifically these words (verified against your actual data)

I checked your raw emails, and the reason isn't subtle — it's literal repetition:

4 separate emails contain almost the exact sentence: "loan what is the outstanding and foreclosure charge" — just the customer's name changes at the end.
4 separate emails contain: "i want to extend the tenure of my [loan]".
4 separate emails contain: "...closed it now for personal reasons..."

So "loan", "outstanding", "extend", and "reasons" keep showing up inside the same handful of repeated template sentences, just a few words apart. Every time CBOW trains on one of those windows, it nudges these words' vectors toward each other a little. With the same templates repeating dozens of times across the 2500 rows, those nudges add up — which is exactly why they end up as each other's nearest neighbors.
Note 5 — The important caveat this reveals about your dataset

This is a useful diagnostic, not just a Word2Vec quirk: real-world Word2Vec on a large, varied corpus usually groups words by meaning (e.g. "loan" near "mortgage", "credit", "EMI"). Here, several of "loan"'s neighbors are really showing up because your dataset has heavily repeated sentence templates (same complaint, different name at the end) — common in synthetic/generated datasets. CBOW is faithfully learning "what tends to sit near this word in this corpus," and in your corpus that's dominated by template repetition rather than diverse natural language.
Note 6 — Why this matters for your fraud-detection model

If you train embeddings on this corpus and feed them into a classifier, the model is partly learning "which template was used" rather than deeper semantic meaning. That's not necessarily bad for classification (templates may correlate with label), but it's worth knowing — it explains why neighbors look like co-occurring template words rather than true synonyms, and it means embeddings learned here probably won't generalize well to emails written in genuinely different phrasing.